<a href="https://colab.research.google.com/github/SandraQA69/ChalengeOneDataScience/blob/main/Version2_Chalenge_TelecomX.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📌 Carga y normalización de datos


In [1]:
import pandas as pd
import json

# Cargar el archivo JSON
with open('/content/TelecomX_Data.json') as f:
    data = json.load(f)

# Convertir a DataFrame y normalizar columnas anidadas
df = pd.json_normalize(data)
df.head()


FileNotFoundError: [Errno 2] No such file or directory: '/content/TelecomX_Data.json'

# 🧹 Limpieza


In [ ]:
# Mostrar info general
df.info()

# Revisar columnas y valores nulos
print(df.isnull().sum())

# Convertir valores de 'Charges.Total' a numérico
df['account.Charges.Total'] = pd.to_numeric(df['account.Charges.Total'], errors='coerce')

# Convertir 'Churn' a binario (1 = Yes, 0 = No)
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})


## Revisa inconsistencias


In [ ]:
import pandas as pd
import json

# Cargar el JSON y normalizar si aún no lo hiciste
with open('/TelecomX_Data.json') as f:
    data = json.load(f)

df = pd.json_normalize(data)

# Convertir columnas clave si es necesario
df['account.Charges.Total'] = pd.to_numeric(df['account.Charges.Total'], errors='coerce')
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})

# 1️⃣ Revisar tipos de datos
print("\n🔍 Tipos de datos:")
print(df.dtypes)

# 2️⃣ Valores nulos
print("\n📉 Valores nulos:")
print(df.isnull().sum())

# 3️⃣ Cadenas vacías
print("\n⚠️ Celdas con cadenas vacías:")
for col in df.columns:
    if df[col].dtype == 'object':
        empty_count = (df[col].str.strip() == '').sum()
        if empty_count > 0:
            print(f"{col}: {empty_count} vacíos")

# 4️⃣ Valores únicos de columnas importantes (para detectar errores)
print("\n🔎 Valores únicos en columnas clave:")
print("Churn:", df['Churn'].unique())
print("InternetService:", df['internet.InternetService'].unique())
print("Contract:", df['account.Contract'].unique())

# 5️⃣ Duplicados por customerID
duplicates = df.duplicated(subset='customerID')
print(f"\n🧬 Registros duplicados por customerID: {duplicates.sum()}")


## Limpiandoo la Base


In [ ]:
# Eliminar filas donde no sabemos si hubo churn
df_clean = df.dropna(subset=['Churn'])

# Opcional: convertir NaN en 'Total Charges' a 0 si corresponde
# Solo si 'tenure' es 0 (clientes nuevos)
df_clean['account.Charges.Total'] = df_clean['account.Charges.Total'].fillna(
    df_clean.apply(lambda row: 0 if row['customer.tenure'] == 0 else row['account.Charges.Total'], axis=1)
)

# Verificar de nuevo
print("Nulos después de limpiar:")
print(df_clean.isnull().sum())


In [ ]:
df_clean

## Manejo de inconsistencias


In [ ]:
import pandas as pd
import json

# --- Cargar JSON y normalizar ---
with open('/TelecomX_Data.json') as f:
    data = json.load(f)

df = pd.json_normalize(data)

# --- Conversión de tipos ---
df['account.Charges.Total'] = pd.to_numeric(df['account.Charges.Total'], errors='coerce')
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})  # Convertir a 1/0

# --- Limpieza de valores especiales ---
servicio_cols = [
    'internet.OnlineSecurity', 'internet.OnlineBackup',
    'internet.DeviceProtection', 'internet.TechSupport',
    'internet.StreamingTV', 'internet.StreamingMovies',
    'phone.MultipleLines'
]

# Reemplazar valores como "No internet service" por "No"
for col in servicio_cols:
    df[col] = df[col].replace({'No internet service': 'No', 'No phone service': 'No'})

# --- Manejo de valores nulos ---
# 1. Eliminar registros sin información de churn
df = df.dropna(subset=['Churn'])

# 2. Rellenar valores nulos en 'Total Charges' solo si tenure = 0
df['account.Charges.Total'] = df.apply(
    lambda row: 0 if pd.isna(row['account.Charges.Total']) and row['customer.tenure'] == 0 else row['account.Charges.Total'],
    axis=1
)

# Verificar nuevamente nulos
print("✅ Nulos restantes:")
print(df.isnull().sum())

# Mostrar dataset limpio
df.head()


# Columnas de cuentas diarias

In [ ]:
import pandas as pd
import json

# Cargar los datos desde el archivo JSON
with open('/TelecomX_Data.json', 'r') as f:
    data = json.load(f)

# Convertir a DataFrame
df = pd.json_normalize(data)

# Renombrar columnas para mayor claridad
df = df.rename(columns={
    'account.Charges.Monthly': 'MonthlyCharges',
    'account.Charges.Total': 'TotalCharges'
})

# Crear la columna Cuentas_Diarias
df['Cuentas_Diarias'] = df['MonthlyCharges'] / 30

# Mostrar las primeras filas para verificar
print(df[['customerID', 'MonthlyCharges', 'Cuentas_Diarias']].head())

## Etapa de traasnformación y estandarización de datos
# Convertimos texto como "Male", "Female", "Month-to-month", etc., en números.

In [ ]:
# Variables binarias (Sí / No o Male / Female)
binarias = ['customer.gender', 'customer.Partner', 'customer.Dependents',
            'phone.PhoneService', 'phone.MultipleLines',
            'account.PaperlessBilling']

for col in binarias:
    df[col] = df[col].map({'Yes': 1, 'No': 0, 'Male': 1, 'Female': 0})

# Otras variables categóricas (más de 2 valores) → One-Hot Encoding
categoricas = ['internet.InternetService', 'account.Contract', 'account.PaymentMethod']
df = pd.get_dummies(df, columns=categoricas, drop_first=True)


#Estandarizar variables numéricas


In [ ]:
from sklearn.preprocessing import StandardScaler
import numpy as np

# Primero, forzar la conversión de columnas numéricas
for col in ['MonthlyCharges', 'TotalCharges']:
    df[col] = df[col].replace(' ', np.nan)  # reemplazar espacios vacíos
    df[col] = pd.to_numeric(df[col], errors='coerce')  # convertir a número

# Eliminar filas con NaN en esas columnas si las hay
df = df.dropna(subset=['MonthlyCharges', 'TotalCharges'])

# Escalar variables numéricas
escalar_cols = ['customer.tenure', 'MonthlyCharges', 'TotalCharges']
scaler = StandardScaler()
df[escalar_cols] = scaler.fit_transform(df[escalar_cols])

# Confirmar que todo está bien
print("✅ Variables estandarizadas:")
print(df[escalar_cols].head())




In [ ]:
print(df.columns)


## 📊 Análisis Exploratorio (EDA) inicial

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Distribución de churn
sns.countplot(x='Churn', data=df)
plt.title("Distribución de Cancelaciones (Churn)")
plt.show()

# Churn por tipo de contrato
sns.countplot(x='account.Contract', hue='Churn', data=df)
plt.title("Churn por Tipo de Contrato")
plt.xticks(rotation=45)
plt.show()

# Correlación entre variables numéricas
numeric_df = df.select_dtypes(include=['number'])
sns.heatmap(numeric_df.corr(), annot=True, cmap='coolwarm')
plt.title("Matriz de Correlación")
plt.show()
